In [ ]:
from toy import *
from pathlib import Path

### Generate underlying data

In [ ]:
group_sizes = [100, 50, 80, 70]
pis = [
    [0.75, 0.10, 0.10, 0.05],
    [0.30, 0.55, 0.10, 0.05],
    [0.18, 0.02, 0.20, 0.60],
    [0.08, 0.12, 0.10, 0.70],
]

# generate truth + observed data
Q_true, P_true, X, groups = generate_toy_truth_and_X(
    group_sizes,
    pis,
    F=200,
    concentration_Q=20,
    concentration_P=0.5,
    n_counts=200,
    seed_data=3,
)


In [ ]:
# one hot encode ground truth group labels
Q_truth = np.zeros((X.shape[0], len(group_sizes)))
start = 0
for i, size in enumerate(group_sizes):
    Q_truth[start : start + size, i] = 1.0
    start += size
# save ground truth
truth_dir = Path("../data/toy") 
truth_dir.mkdir(parents=True, exist_ok=True)
np.savetxt(truth_dir / "X.txt", X, delimiter=",")
np.savetxt(truth_dir / "ground_truth.Q", Q_truth)

### Perform mixed-membership clustering NMF

In [ ]:
method = "nmf"   # "nmf" or "lda"
# estimate Q/P multiple times (different seeds/inits)
Q_runs = []
P_runs = []
for K in [3,4,5]:
    Q_runs_K, P_runs_K, perms = estimate_QP_runs(
        X, K,
        n_runs=5,
        base_seed_fit=0,
        method=method,        # or "lda"
        shuffle_cols=True,
    )
    # append method to names
    Q_runs_K = [(f"{method}_{name}", Q) for name, Q in Q_runs_K]
    P_runs_K = [(f"{method}_{name}", P) for name, P in P_runs_K]
    Q_runs.extend(Q_runs_K)
    P_runs.extend(P_runs_K)

# # visualize a random subset of estimated Q's
# n_rdm = 5
# rdm_indices = np.random.choice(len(Q_runs), size=n_rdm, replace=False)
# fig, axes = plt.subplots(nrows=len(rdm_indices), ncols=1, figsize=(8, 2*n_rdm), dpi=150)
# # iterate through selected Q_items and corresponding axes
# for ax, (name, Q) in zip(axes, [Q_runs[i] for i in rdm_indices]):
#     print(name, Q.shape, np.allclose(Q.sum(axis=1), 1))
#     plot_toy_Q(Q, title="", figsize=(8,3), ax=ax)
# fig.tight_layout()


In [ ]:
# save
toy_dir = Path("../data/toy/clustering") / method
toy_dir.mkdir(parents=True, exist_ok=True)
save_toy_QP_outputs(
    toy_dir,
    Q_true=None,
    P_true=None,
    X=X,
    groups=groups,
    Q_runs=Q_runs,
    P_runs=P_runs,
    perms=None,
    overwrite=True,
)


### Perform mixed-membership clustering using LDA

In [ ]:
method = "lda"   # "nmf" or "lda"
# estimate Q/P multiple times (different seeds/inits)
Q_runs = []
P_runs = []
for K in [3,4,5]:
    Q_runs_K, P_runs_K, perms = estimate_QP_runs(
        X, K,
        n_runs=5,
        base_seed_fit=0,
        method=method,        
        shuffle_cols=True,
    )
    # append method to names
    Q_runs_K = [(f"{method}_{name}", Q) for name, Q in Q_runs_K]
    P_runs_K = [(f"{method}_{name}", P) for name, P in P_runs_K]
    Q_runs.extend(Q_runs_K)
    P_runs.extend(P_runs_K)


In [ ]:
# save
toy_dir = Path("../data/toy/clustering") / method
toy_dir.mkdir(parents=True, exist_ok=True)
save_toy_QP_outputs(
    toy_dir,
    Q_true=None,
    P_true=None,
    X=X,
    groups=groups,
    Q_runs=Q_runs,
    P_runs=P_runs,
    perms=None,
    overwrite=True,
)

### Perform hard clustering using K-means

In [ ]:
method = "kmeans"
Q_runs = []
for K in [3, 4, 5]:
    Q_runs_K, perms = estimate_Q_runs_kmeans(
        X, K,
        n_runs=5,
        base_seed_fit=0,
        shuffle_cols=True,
    )
    # names already include "kmeans"
    Q_runs.extend(Q_runs_K)


In [ ]:
# save
toy_dir = Path("../data/toy/clustering") / method
toy_dir.mkdir(parents=True, exist_ok=True)
save_toy_QP_outputs(
    toy_dir,
    Q_true=None,
    P_true=None,
    X=X,
    groups=groups,
    Q_runs=Q_runs,
    P_runs=None,
    perms=None,
    overwrite=True,
)
